# Analisis exploratorio de datos
**Objetivos**: 

El objetivo de esta primera sección es comprender y describir resultados con los datos presentes es decir poder afirmar qué resultados estamos obteniendo solo con la información que tenemos presentes intentaremos no extendernos demasiado y responder solo a algunas preguntas principales ya que esto es una tarea relativamente sencilla y nuestro objetivo es enfocarnos más en las cuestiones más complejas por lo que describimos a grandes rasgos cómo funciona esta sección y dejando varias preguntas que podríamos responder en el futuro

Primero nos gustaría responder a las siguientes preguntas:
* en qué regiones se concentra la mayor cantidad de ventas 
* cuáles son los artículos más vendidos 
* cuántas órdenes tenemos distribuidas por estatus además de cuál es la puntuación promedio de las reviews 
* cómo se distribuyen las calificaciones de acuerdo a el tipo de producto que estamos vendiendo

* Customers → ¿Quién compró? (ciudad, estado, ZIP)
* Geolocation → ¿Dónde está cada ZIP? (coordenadas para mapas)
* Orders → ¿Cuándo y en qué estado está cada pedido? (fechas de compra, entrega, estimada)
* Order Items → ¿Qué se compró exactamente? (producto, vendedor, precio, flete por ítem)
* Payments → ¿Cómo y cuánto se pagó? (tarjeta, boleto, cuotas)
* Reviews → ¿Qué opinaron los clientes? (nota 1-5 + comentarios)
* Products → ¿Qué productos hay? (categoría, tamaño, peso)
* Category Translation → Traduce categorías PT → EN
* Sellers → ¿Quién vendió? (localización del vendedor)

In [6]:
import os, sys
sys.path.insert(0, r"d:\Supply Chain Resilience")
import matplotlib.pyplot as plt
import duckdb 
import pandas as pd
import matplotlib
from src.data_save import DataSave

In [7]:
ruta_datos_oro = DataSave(ruta="Dataset/03 Olist Gold")
tablas = ruta_datos_oro.cargar_todo()

In [8]:
def resumen_tabla(dataframes: dict) -> pd.DataFrame:
  filas = []
  for name, df in dataframes.items():
    null_cols = df.columns[df.isnull().any()].tolist()
    filas.append({
        "dataset":         name,
        "n_filas":         df.shape[0],
        "n_columnas":      df.shape[1],
        "val_nulos":     df.isnull().sum().sum(),
        "qty_null_columns": len(null_cols),
        "null_columns":    ", ".join(null_cols) if null_cols else ""
    })
  return pd.DataFrame(filas)

def styling_pandas(df: dict) -> pd.DataFrame:
  # Get the names of the numeric columns
  num_cols_names = df.select_dtypes(include='number').columns.tolist()
  styled = (
      df.style
      .background_gradient(
          cmap="Blues",
          subset=num_cols_names # Pass the list of column names here
      )
      .set_properties(**{
          "font-size": "12px",
          "border": "1px solid #ddd",
          "text-align": "center"
      })
      .set_table_styles([{
          "selector": "th",
          "props": [
              ("background-color", "#2c3e50"),
              ("color", "white"),
              ("font-size", "13px"),
              ("text-align", "center"),
              ("padding", "8px")
          ]
      }])
      .format({
          "n_filas":          "{:,}", # Corrected column name
          "val_nulos":     "{:,}", # Corrected column name
          "qty_null_columns": "{:,}"
      })
  )
  return styled

In [9]:
def plot_top_n(
    data,
    x_axis: str = None,
    y_axis: str = None,
    top_n: int = 10,
    titulo: str = "Distribución (Top {top_n})",
    xlabel: str = None,
    ylabel: str = "Cantidad",
    figsize: tuple = (12, 7),
    cmap_name: str = "Purples",
    rotacion_xticks: int = 45,
    mostrar_otros: bool = True,
    mostrar_porcentaje: bool = True,
) -> None:


    # ── Validaciones ─────────────────────────────────
    if data is None:
        raise ValueError("'data' no puede ser None.")

    if isinstance(data, pd.DataFrame):
        if y_axis is None:
            raise ValueError("Especificar 'columna'.")
        if y_axis not in data.columns:
            raise ValueError(f"La columna '{y_axis}' no existe en el DataFrame.")

        # ── Usar columna_index como índice si se especifica ──
        if x_axis:
            if x_axis not in data.columns:
                raise ValueError(f"La columna '{x_axis}' no existe en el DataFrame.")
            data = data.set_index(x_axis)

        conteos = data[y_axis]

    elif isinstance(data, pd.Series):
        conteos = data

    else:
        raise TypeError("'data' debe ser pd.Series o pd.DataFrame.")

    # ── Validar datos vacíos ──────────────────────────
    if conteos.empty:
        print("No hay datos para graficar.")
        return

    # ── Limitar top_n al tamaño real ──────────────────
    top_n = min(top_n, len(conteos))
    top   = conteos.head(top_n).copy()

    # ── Agrupar el resto en Otros ─────────────────────
    if mostrar_otros:
        otros = conteos.iloc[top_n:].sum()
        if otros > 0:
            top["Otros"] = otros

    # ── Porcentajes ───────────────────────────────────
    porcentajes = (top / top.sum() * 100).round(1)

    # ── Etiquetas ─────────────────────────────────────
    if mostrar_porcentaje:
        etiquetas = [f"{int(v):,}\n({p}%)" for v, p in zip(top, porcentajes)]
    else:
        etiquetas = [f"{int(v):,}" for v in top]

    # ── Gráfico ───────────────────────────────────────
    fig, ax = plt.subplots(figsize=figsize)

    n_barras = len(top)
    colores  = matplotlib.colormaps[cmap_name](np.linspace(0.85, 0.35, n_barras))

    bars = ax.bar(
        x      = range(n_barras),
        height = top.values,
        color  = colores,
        width  = 0.6
    )

    # ── Etiquetas sobre barras ────────────────────────
    margen = top.max() * 0.015
    for bar, etiqueta in zip(bars, etiquetas):
        ax.text(
            x          = bar.get_x() + bar.get_width() / 2,
            y          = bar.get_height() + margen,
            s          = etiqueta,
            ha         = "center",
            va         = "bottom",
        )

    # ── Eje X ─────────────────────────────────────────
    ax.set_xticks(range(n_barras))
    ax.set_xticklabels(
        top.index,
        rotation = rotacion_xticks,
        ha       = "right",
        fontsize = 10
    )

    # ── Margen superior para etiquetas ───────────────
    ax.set_ylim(0, top.max() * 1.18)

    # ── Estilo ────────────────────────────────────────
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.show()

In [10]:
resumen = resumen_tabla(tablas)
styling_pandas(resumen)

ImportError: Missing optional dependency 'Jinja2'. DataFrame.style requires jinja2. Use pip or conda to install Jinja2.

In [ ]:
def resumen_columnas(tabla: dict) -> pd.DataFrame:
    filas = []
    for name, df in tablas.items():
      for col in df.columns:
          val_null = df[col].isnull().sum()
          dtype = str(df[col].dtype)

          filas.append({
              "nombre_df":  name,
              "nombre_col":  col,
              "val_null":  val_null,
              "tipo":  dtype,
          })

    return pd.DataFrame(filas)

In [ ]:
resumen_columnas(tablas)

In [ ]:
# hacemos la conexion con db guardo en con
con = duckdb.connect()
# registramos cada tabla
for nombre, df in tablas.items():
    con.register(nombre, df)
# usamos la conexion especifica y no la global duckdb
def query(sql: str, conexion=con) -> pd.DataFrame:
    try:
        resultado = conexion.query(sql).df()
        return resultado
    except Exception as e:
        print(f"Error en la query: {e}")
        return pd.DataFrame()

In [ ]:
query_1 = """
SELECT
  customer_city,
  count(customer_id) as cantidad
FROM dim_customers
GROUP BY customer_city
oRDER BY cantidad DESC
"""

In [ ]:
ventas_x_ciudad = query(query_1)

In [ ]:
plot_top_n(
    ventas_x_ciudad,
    y_axis       = "cantidad",
    x_axis   = "customer_city",
    top_n         = 10,
    xlabel        = "Ciudad",
    ylabel        = "Cantidad de ventas"
)

In [ ]:
query_2 = """
SELECT
  order_status,
  COUNT(order_status) AS Cantidad
FROM dim_orders
GROUP BY 
  order_status
ORDER BY 
  Cantidad DESC
"""
resp_2 = query(query_2)

In [ ]:
plot_top_n(resp_2,
           x_axis= 'order_status',
           y_axis= 'Cantidad',
           xlabel= 'Estatus de Orden',
           ylabel= 'Cantidad de ordenes',
           top_n=len(resp_2),
           titulo = 'Distribucion de ordenes por status')

In [ ]:
query_3 = """
SELECT
    dp.product_category_name,
    SUM(dot.price)          AS Precio,
    SUM(dot.freight_value)  AS Costo_Envio
FROM dim_order_items dot
JOIN dim_products dp
    ON dot.product_id = dp.product_id
GROUP BY 
    dp.product_category_name
ORDER BY Precio DESC
"""
resp_3 = query(query_3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ax.bar(resp_3['product_category_name'][:10], resp_3['Precio'][:10],color="steelblue", width=0.6, alpha=0.7, label="Ventas del producto")
ax.plot(resp_3['product_category_name'][:10], resp_3['Costo_Envio'][:10],color="salmon", alpha=0.7, label="Costo de envio")

ax.legend()
ax.set_xticks(range(10))
ax.set_xticklabels(
    labels   = resp_3['product_category_name'][:10],
    rotation = 45,
    ha       = "right",
    fontsize = 10
)
plt.tight_layout()
plt.show()

In [ ]:
promedio_review = """
SELECT
    AVG(review_score) AS calificacion_promedio,
    MAX(review_score) AS calificacion_minima,
    MIN(review_score) AS calificacion_maxima
FROM dim_orders_reviews
-- JOIN 
-- GROUP 
-- ORDER BY Precio DESC
"""
promedio_review = query(promedio_review)
promedio_review

In [ ]:
# Cuál es la puntuación de las reviews en promedio
query_5 = """
SELECT
    dp.product_category_name        AS categoria,
    COUNT(DISTINCT f.order_id)      AS total_ordenes,
    ROUND(AVG(r.review_score), 2)   AS calificacion_promedio,
    MIN(r.review_score)             AS calificacion_minima,
    MAX(r.review_score)             AS calificacion_maxima
FROM fact_sales f
JOIN dim_products dp
    ON f.product_id = dp.product_id
LEFT JOIN dim_orders_reviews r
    ON f.order_id = r.order_id
GROUP BY
    dp.product_category_name
ORDER BY calificacion_promedio ASC

"""
respo_5= query(query_5)
respo_5

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

ax.bar(respo_5['categoria'][:10], respo_5['calificacion_promedio'][:10],color="steelblue", width=0.6, label="Ventas del producto")
ax.axhline( y = promedio_review['calificacion_promedio'].values[0], color = "orange", linestyle = "--", linewidth = 1.5, label = "Promedio general")
ax.plot(respo_5['categoria'][:10], respo_5['calificacion_minima'][:10], color="salmon",label="Calificación minima")
ax.plot(respo_5['categoria'][:10], respo_5['calificacion_maxima'][:10], color="green",label="Calificación minima")

ax.legend()
ax.set_xticks(range(10))
ax.set_xticklabels(
    labels   = respo_5['categoria'][:10],
    rotation = 45,
    ha       = "right",
    fontsize = 10
)
plt.tight_layout()
plt.show()
